In [0]:
%run ./Framework

In [0]:
dbutils.widgets.text("pipeline_name", "")
pipeline_name = dbutils.widgets.get("pipeline_name")

In [0]:
silver_job = Silver.from_conf(pipeline_name)


In [0]:
bronze_df = silver_job.read_from_bronze_sk()

if bronze_df is None:
    print("No new rows in Bronze. Skipped.")
else:
    invalid_df = silver_job.invalid_records(bronze_df)
    key_null_df = silver_job.key_null_record(bronze_df)
    duplicate_df = silver_job.duplicate_record(bronze_df, key_null_df)

    bad_record_df = silver_job.union_bad_rec(invalid_df, key_null_df, duplicate_df)

    final_df = silver_job.final_result(bronze_df, bad_record_df)
    final_df = silver_job.apply_business_rules(final_df)

    print(silver_job.load_final_result(final_df))
    print(silver_job.load_bad_rec_to_table(bad_record_df))